# 基于OSM、GEM和空间负荷的中国电网一日机组组合Demo

本notebook把三个上游实验统一为PyPSA数据模型，并求解一个24小时基础UC。模型
包含基态节点平衡、线性化AC潮流、线路容量、机组启停、最小出力、爬坡、最小
开停机时间、启动/停机成本、储能能量平衡和高惩罚失负荷。它不生成也不加入
N-1、N-k或其他事故后安全约束。

## 1. 运行环境与集中配置

本次容量因子和线路额定容量均为框架验证假设，不是中国逐项目量测。`operating`
采用当前GEM版本状态，因此2024年负荷与机组状态不是严格的历史同截面。

In [ ]:
from pathlib import Path
import re

from IPython.display import display
from matplotlib.patches import Patch
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
import xarray as xr
import pypsa

uc_output_dir = Path("outputs")
uc_output_dir.mkdir(exist_ok=True)
uc_line_assumptions_path = Path("config/uc_line_assumptions.csv")
uc_reference_path = Path(
    "../references/2023-ieee trans power syst-chen-"
    "security-constrained unit commitment for electricity market modeling "
    "solution methods and future challenges.pdf"
)

uc_day = pd.Timestamp("2024-08-01")
uc_generator_statuses = ("operating",)
uc_commitment_aggregation = "node_type_largest_clusters"
uc_max_committable_clusters_per_type = 5
uc_enforce_ramp_constraints = False
uc_storage_statuses = ("Operational",)
uc_base_voltage_kv = 500.0
uc_line_capacity_multiplier = 1.0
uc_load_shedding_cost_eur_per_mwh = 10_000.0
uc_solver = "highs"
uc_solver_time_limit_s = 600
uc_solver_mip_gap = 0.01
uc_figure_size = (14, 7)
uc_figure_dpi = 300

uc_resource_profile_mode = "fixed"  # fixed | cf_file
uc_capacity_factor_path = Path(
    f"outputs/capacity_factors_{uc_day.year}.nc"
)
uc_cf_variable_by_type = {
    "onshore_wind": "onshore_wind",
    "offshore_wind": "offshore_wind",
    "utility_scale_solar": "utility_scale_solar",
    "reservoir_hydropower": "reservoir_hydropower",
    "run_of_river_hydropower": "run_of_river_hydropower",
    "other_hydropower": "other_hydropower",
}
uc_fixed_resource_availability_pu = {
    "utility_scale_solar": 0.25,
    "solar_thermal": 0.40,
    "onshore_wind": 0.35,
    "offshore_wind": 0.45,
    "reservoir_hydropower": 0.55,
    "run_of_river_hydropower": 0.45,
    "other_hydropower": 0.50,
}
uc_storage_defaults = {
    "pumped_storage": (8.0, 0.90, 0.90, 0.001),
    "lithium_ion_storage": (4.0, 0.95, 0.95, 0.001),
    "flow_battery_storage": (6.0, 0.85, 0.85, 0.001),
    "compressed_air_storage": (8.0, 0.75, 0.75, 0.001),
    "thermal_storage": (8.0, 0.80, 0.80, 0.005),
    "sodium_battery_storage": (4.0, 0.90, 0.90, 0.001),
    "capacitor_storage": (1.0, 0.95, 0.95, 0.010),
    "other_storage": (4.0, 0.85, 0.85, 0.002),
}

## 2. 执行并校验上游实验

按顺序运行OSM、GEM和LOAD。`%run`共享当前IPython namespace，因此通用变量和
下划线中间变量可能被后执行的notebook覆盖；UC只读取名称明确的最终对象，并在
运行前后检查所有`uc_`配置没有被上游重定义。GEM与LOAD在已有完整OSM对象时直接
复用，不重复构造拓扑。


In [ ]:
%%capture
_uc_configuration_before = {
    name: value
    for name, value in globals().items()
    if name.startswith("uc_")
}
%run ./osm_exp.ipynb
%run ./gem_exp.ipynb
%run ./load_exp.ipynb
_uc_overwritten_configuration = [
    name for name, value in _uc_configuration_before.items()
    if name not in globals() or globals()[name] is not value
]
if _uc_overwritten_configuration:
    raise RuntimeError(
        "上游notebook覆盖了UC配置变量: "
        f"{_uc_overwritten_configuration}"
    )


In [ ]:
_required_inputs = {
    "grid_topology", "grid_nodes_gdf", "grid_branches_gdf",
    "station_nodes_gdf", "gem_grid_matches", "gem_unit_parameters",
    "storage_grid_matches", "station_hourly_load",
}
_missing_inputs = _required_inputs.difference(globals())
if _missing_inputs:
    raise RuntimeError(f"上游notebook缺少对象: {sorted(_missing_inputs)}")
if not nx.is_connected(grid_topology):
    raise RuntimeError("UC输入grid_topology不是连通图。")

if uc_resource_profile_mode not in {"fixed", "cf_file"}:
    raise ValueError(
        "uc_resource_profile_mode只能是fixed或cf_file。"
    )
if (
    uc_resource_profile_mode == "cf_file"
    and not uc_capacity_factor_path.exists()
):
    raise FileNotFoundError(
        f"未找到{uc_capacity_factor_path}，请先运行cf_exp.ipynb。"
    )

_snapshots = pd.date_range(uc_day, periods=24, freq="h")
if not _snapshots.isin(station_hourly_load.index).all():
    raise ValueError(f"站点负荷不完整覆盖{uc_day.date()}的24小时。")

uc_input_summary = pd.Series({
    "pypsa_version": pypsa.__version__,
    "simulation_day": str(uc_day.date()),
    "resource_profile_mode": uc_resource_profile_mode,
    "topology_nodes": grid_topology.number_of_nodes(),
    "topology_branches": grid_topology.number_of_edges(),
    "station_load_nodes": station_hourly_load.shape[1],
    "gem_parameter_rows": len(gem_unit_parameters),
    "gem_mapping_rows": len(gem_grid_matches),
}, name="value")
display(uc_input_summary)

## 3. 统一电源与储能输入

`gem_unit_parameters`是一行一个GEM unit/phase的技术经济宽表，但参数来自GEM
类型/容量、PyPSA technology-data、Dispa-SET和NREL代理的联合映射，不是GEM
原始表自带的完整机组参数。`gem_grid_matches`保留空间记录和候选站点，两者按
`GEM unit/phase ID`一对一合并为`uc_generation_units`。

UC不再重复维护电源类型清单，而读取`technical_committable`判断是否具备启停
属性。为控制全国MILP规模，每类只将容量最大的前
`uc_max_committable_clusters_per_type`个节点组合设为committable，其余仍可连续
调度。所有必要参数在本cell逐字段检查，缺失时直接报错，不再设置第二层边际成本
或启动成本fallback。


In [ ]:
_parameter_columns = [
    "GEM unit/phase ID",
    *[
        column for column in gem_unit_parameters
        if column.startswith(("technical_", "economic_"))
        or column in {
            "investment_eur_per_kw", "fixed_om_percent_per_year",
            "variable_om_eur_per_mwh", "efficiency",
            "lifetime_years", "fuel_eur_per_mwh_th",
            "co2_t_per_mwh_th", "fixed_om_eur_per_kw_year",
            "startup_cost_eur", "marginal_cost_eur_per_mwh",
        }
    ],
]
uc_generation_units = (
    gem_grid_matches.loc[
        gem_grid_matches["Status"].isin(uc_generator_statuses)
        & gem_grid_matches["model_inclusion"]
        & gem_grid_matches["node_uid"].isin(grid_topology)
        & gem_grid_matches["Capacity (MW)"].gt(0)
    ]
    .merge(
        gem_unit_parameters[_parameter_columns],
        on="GEM unit/phase ID",
        how="left",
        validate="one_to_one",
    )
)

_required_uc_parameters = [
    "technical_committable",
    "technical_minimum_output_pu",
    "technical_ramp_up_pu_per_hour",
    "technical_ramp_down_pu_per_hour",
    "technical_minimum_up_time_h",
    "technical_minimum_down_time_h",
    "economic_shutdown_cost_eur_per_mw",
    "startup_cost_eur",
    "marginal_cost_eur_per_mwh",
]
_missing_parameters = (
    uc_generation_units[_required_uc_parameters].isna()
    .groupby(uc_generation_units["generation_type"])
    .sum()
)
if _missing_parameters.to_numpy().sum():
    display(_missing_parameters.loc[_missing_parameters.any(axis=1)])
    raise ValueError(
        "GEM技术经济映射仍有缺失，UC不再静默使用边际成本或启停参数兜底。"
    )

_weighted_fields = [
    "technical_minimum_output_pu",
    "technical_ramp_up_pu_per_hour",
    "technical_ramp_down_pu_per_hour",
    "marginal_cost_eur_per_mwh",
]
for _field in _weighted_fields:
    uc_generation_units[f"_{_field}_mw"] = (
        uc_generation_units[_field]
        * uc_generation_units["Capacity (MW)"]
    )
uc_generation_units["_efficiency_mw"] = (
    uc_generation_units["efficiency"].fillna(1.0)
    * uc_generation_units["Capacity (MW)"]
)
uc_generation_units["_co2_t_per_mwh_th_mw"] = (
    uc_generation_units["co2_t_per_mwh_th"].fillna(0.0)
    * uc_generation_units["Capacity (MW)"]
)
uc_generation_units["_shutdown_cost_eur"] = (
    uc_generation_units["economic_shutdown_cost_eur_per_mw"]
    * uc_generation_units["Capacity (MW)"]
)

_generation = uc_generation_units[
    ~uc_generation_units["generation_type"].eq("pumped_storage")
].copy()
_generation["_cluster_key"] = (
    _generation["node_uid"].astype(str)
    + "|"
    + _generation["generation_type"]
)
uc_generators = (
    _generation.groupby(
        ["_cluster_key", "node_uid", "generation_type"],
        as_index=False,
    )
    .agg(
        p_nom_mw=("Capacity (MW)", "sum"),
        source_units=("GEM unit/phase ID", "nunique"),
        source_nodes=("node_uid", "nunique"),
        commitment_eligible=("technical_committable", "all"),
        startup_cost_eur=("startup_cost_eur", "sum"),
        shutdown_cost_eur=("_shutdown_cost_eur", "sum"),
        minimum_up_time_h=("technical_minimum_up_time_h", "max"),
        minimum_down_time_h=("technical_minimum_down_time_h", "max"),
        _efficiency_mw=("_efficiency_mw", "sum"),
        _co2_t_per_mwh_th_mw=(
            "_co2_t_per_mwh_th_mw", "sum"
        ),
        **{
            f"_{field}_mw": (f"_{field}_mw", "sum")
            for field in _weighted_fields
        },
    )
)
for _field in [
    *_weighted_fields, "efficiency", "co2_t_per_mwh_th"
]:
    uc_generators[_field] = (
        uc_generators[f"_{_field}_mw"]
        / uc_generators["p_nom_mw"]
    )
uc_generators["committable"] = (
    uc_generators["commitment_eligible"]
    & uc_generators.groupby("generation_type")["p_nom_mw"]
    .rank(method="first", ascending=False)
    .le(uc_max_committable_clusters_per_type)
)
uc_generators["p_min_pu"] = np.where(
    uc_generators["committable"],
    uc_generators["technical_minimum_output_pu"],
    0.0,
)
uc_generators["ramp_limit_up_pu"] = np.where(
    uc_generators["committable"],
    uc_generators[[
        "technical_ramp_up_pu_per_hour", "p_min_pu"
    ]].max(axis=1),
    uc_generators["technical_ramp_up_pu_per_hour"],
)
uc_generators["ramp_limit_down_pu"] = np.where(
    uc_generators["committable"],
    uc_generators[[
        "technical_ramp_down_pu_per_hour", "p_min_pu"
    ]].max(axis=1),
    uc_generators["technical_ramp_down_pu_per_hour"],
)
uc_generators["p_max_pu"] = (
    uc_generators["generation_type"]
    .map(uc_fixed_resource_availability_pu)
    .fillna(1.0)
)
uc_generators["generator_uid"] = [
    f"generator_{index:05d}" for index in range(len(uc_generators))
]
uc_generators = uc_generators.drop(
    columns=[
        "_cluster_key",
        *[f"_{field}_mw" for field in [
            *_weighted_fields, "efficiency", "co2_t_per_mwh_th"
        ]],
    ]
).set_index("generator_uid")

_pumped = (
    uc_generation_units[
        uc_generation_units["generation_type"].eq("pumped_storage")
    ]
    .groupby("node_uid", as_index=False)
    .agg(
        p_nom_mw=("Capacity (MW)", "sum"),
        source_projects=("GEM location ID", "nunique"),
    )
    .assign(storage_type="pumped_storage")
)
_other_storage = (
    storage_grid_matches.loc[
        storage_grid_matches["storage_status"].isin(uc_storage_statuses)
        & storage_grid_matches["model_inclusion"]
        & storage_grid_matches["node_uid"].isin(grid_topology)
        & storage_grid_matches["power_capacity_mw"].gt(0),
        [
            "node_uid", "storage_project_id", "storage_type",
            "power_capacity_mw", "energy_capacity_mwh",
        ],
    ]
    .rename(columns={"power_capacity_mw": "p_nom_mw"})
)
_other_storage["source_projects"] = 1
uc_storage_units = pd.concat(
    [
        _pumped,
        _other_storage.drop(columns="storage_project_id"),
    ],
    ignore_index=True,
)
uc_storage_units["reported_max_hours"] = (
    uc_storage_units.get("energy_capacity_mwh")
    / uc_storage_units["p_nom_mw"]
)
for _storage_type, (_hours, _store_eff, _dispatch_eff, _loss) in (
    uc_storage_defaults.items()
):
    _selected = uc_storage_units["storage_type"].eq(_storage_type)
    uc_storage_units.loc[_selected, "max_hours"] = (
        uc_storage_units.loc[_selected, "reported_max_hours"]
        .where(lambda value: value.gt(0))
        .fillna(_hours)
    )
    uc_storage_units.loc[_selected, "efficiency_store"] = _store_eff
    uc_storage_units.loc[_selected, "efficiency_dispatch"] = _dispatch_eff
    uc_storage_units.loc[_selected, "standing_loss"] = _loss
uc_storage_units["storage_uid"] = [
    f"storage_{index:04d}" for index in range(len(uc_storage_units))
]
uc_storage_units = uc_storage_units.set_index("storage_uid")

uc_generation_summary = uc_generators.groupby("generation_type").agg(
    clustered_generators=("p_nom_mw", "size"),
    source_units=("source_units", "sum"),
    capacity_mw=("p_nom_mw", "sum"),
    committable_clusters=("committable", "sum"),
)
display(uc_generation_summary)

## 4. 构造PyPSA统一数据表

| PyPSA表 | 内容 | 上游主键 |
|---|---|---|
| `uc_buses` / `network.buses` | 最大连通图全部station和junction节点 | OSM `node_uid` |
| `uc_lines` / `network.lines` | AC segment支路、阻抗和容量 | OSM `branch_uid` |
| `uc_links` / `network.links` | 可双向调度的DC支路 | OSM `branch_uid` |
| `uc_generators` / `network.generators` | 节点-类型聚合电源及UC参数 | GEM unit IDs汇总 |
| `uc_storage_units` | 抽蓄及DOE储能 | GEM/DOE项目 |
| `uc_loads` / `network.loads_t.p_set` | 站点静态信息和24小时MW负荷 | OSM `node_uid` |

现有OSM将多电压变电站视为可连接不同电压线路的一个节点。为保持这一语义且不
新增变压器，所有PyPSA bus使用500 kV数值基准；每条AC支路的物理阻抗按
\((V_{base}/V_{line})^2\)折算，使其标幺阻抗仍对应原线路电压。

In [ ]:
uc_line_assumptions = pd.read_csv(uc_line_assumptions_path)
uc_buses = grid_nodes_gdf.loc[
    grid_nodes_gdf["node_uid"].isin(grid_topology),
    [
        "node_uid", "node_type", "node_province_name",
        "node_longitude", "node_latitude",
    ],
].copy()
uc_buses["bus"] = [
    f"bus_{index:05d}" for index in range(len(uc_buses))
]
_bus_by_node = uc_buses.set_index("node_uid")["bus"].to_dict()
uc_buses["v_nom_kv"] = uc_base_voltage_kv
uc_buses = uc_buses.set_index("bus")

def _circuit_count(value):
    _numbers = re.findall(r"\d+", str(value))
    return max(int(_numbers[0]), 1) if _numbers else 1


uc_branches = grid_branches_gdf.loc[
    grid_branches_gdf["from_node"].isin(_bus_by_node)
    & grid_branches_gdf["to_node"].isin(_bus_by_node)
].copy()
uc_branches["bus0"] = uc_branches["from_node"].map(_bus_by_node)
uc_branches["bus1"] = uc_branches["to_node"].map(_bus_by_node)
uc_branches["circuits_assumed"] = uc_branches["circuits"].map(
    _circuit_count
)
_assumption_rows = np.abs(
    uc_branches["voltage_kv"].to_numpy()[:, None]
    - uc_line_assumptions["voltage_kv"].to_numpy()[None, :]
).argmin(axis=1)
uc_branches = pd.concat(
    [
        uc_branches.reset_index(drop=True),
        uc_line_assumptions.iloc[_assumption_rows]
        .reset_index(drop=True)
        .add_prefix("assumption_"),
    ],
    axis=1,
)
_voltage_scale = (
    uc_base_voltage_kv / uc_branches["voltage_kv"]
) ** 2
uc_branches["r_ohm_equivalent"] = (
    uc_branches["assumption_r_ohm_per_km"]
    * uc_branches["length_km"]
    * _voltage_scale
    / uc_branches["circuits_assumed"]
)
uc_branches["x_ohm_equivalent"] = (
    uc_branches["assumption_x_ohm_per_km"]
    * uc_branches["length_km"]
    * _voltage_scale
    / uc_branches["circuits_assumed"]
)
uc_branches["s_nom_mva"] = (
    uc_branches["assumption_s_nom_mva_per_circuit"]
    * uc_branches["circuits_assumed"]
    * uc_line_capacity_multiplier
)
uc_branches["pypsa_branch_uid"] = [
    f"branch_{index:05d}" for index in range(len(uc_branches))
]

uc_lines = uc_branches[
    uc_branches["current_type"].eq("AC")
].set_index("pypsa_branch_uid")
uc_links = uc_branches[
    uc_branches["current_type"].eq("DC")
].set_index("pypsa_branch_uid")

_station_load_mw = (
    station_hourly_load.reindex(_snapshots)
    .loc[:, lambda frame: frame.columns.isin(_bus_by_node)]
    * 1000
)
uc_loads = pd.DataFrame({
    "node_uid": _station_load_mw.columns,
    "bus": [_bus_by_node[node] for node in _station_load_mw.columns],
    "peak_load_mw": _station_load_mw.max().to_numpy(),
    "daily_energy_mwh": _station_load_mw.sum().to_numpy(),
})
uc_loads["load_uid"] = [
    f"load_{index:04d}" for index in range(len(uc_loads))
]
uc_loads = uc_loads.set_index("load_uid")
uc_load_profiles = _station_load_mw.copy()
uc_load_profiles.columns = uc_loads.index

_load_shedding = pd.DataFrame({
    "node_uid": uc_loads["node_uid"].to_numpy(),
    "generation_type": "load_shedding",
    "p_nom_mw": uc_loads["peak_load_mw"].to_numpy() * 1.05,
    "source_units": 0,
    "startup_cost_eur": 0.0,
    "shutdown_cost_eur": 0.0,
    "minimum_up_time_h": 0,
    "minimum_down_time_h": 0,
    "ramp_limit_up_pu": np.nan,
    "ramp_limit_down_pu": np.nan,
    "marginal_cost_eur_per_mwh": uc_load_shedding_cost_eur_per_mwh,
    "efficiency": 1.0,
    "co2_t_per_mwh_th": 0.0,
    "committable": False,
    "p_min_pu": 0.0,
    "p_max_pu": 1.0,
}, index=[
    f"load_shedding_{index:04d}" for index in range(len(uc_loads))
])
uc_generators = pd.concat([uc_generators, _load_shedding], axis=0)
uc_generators["bus"] = uc_generators["node_uid"].map(_bus_by_node)
uc_storage_units["bus"] = uc_storage_units["node_uid"].map(_bus_by_node)

uc_generator_availability = pd.DataFrame(
    np.broadcast_to(
        uc_generators["p_max_pu"].to_numpy(),
        (len(_snapshots), len(uc_generators)),
    ).copy(),
    index=_snapshots,
    columns=uc_generators.index,
)
if uc_resource_profile_mode == "cf_file":
    with xr.open_dataset(uc_capacity_factor_path) as _cf_dataset:
        _required_cf_variables = {
            variable for generation_type, variable
            in uc_cf_variable_by_type.items()
            if generation_type in uc_generators["generation_type"].values
        }
        _missing_cf_variables = (
            _required_cf_variables.difference(_cf_dataset.data_vars)
        )
        if _missing_cf_variables:
            raise ValueError(
                "容量因子文件缺少变量: "
                f"{sorted(_missing_cf_variables)}"
            )
        for _generation_type, _variable in (
            uc_cf_variable_by_type.items()
        ):
            _selected = uc_generators[
                "generation_type"
            ].eq(_generation_type)
            if not _selected.any():
                continue
            _selected_generators = uc_generators.index[_selected]
            _generator_buses = uc_generators.loc[
                _selected_generators, "bus"
            ]
            _sampled = _cf_dataset[_variable].sel(
                time=_snapshots,
                x=xr.DataArray(
                    _generator_buses.map(
                        uc_buses["node_longitude"]
                    ).to_numpy(),
                    dims="generator",
                ),
                y=xr.DataArray(
                    _generator_buses.map(
                        uc_buses["node_latitude"]
                    ).to_numpy(),
                    dims="generator",
                ),
                method="nearest",
            ).transpose("time", "generator")
            uc_generator_availability.loc[
                :, _selected_generators
            ] = _sampled.to_numpy()

if (
    uc_generator_availability.isna().any().any()
    or (uc_generator_availability < 0).any().any()
    or (uc_generator_availability > 1).any().any()
):
    raise ValueError("发电可用率必须完整且位于[0, 1]。")

if uc_generators["bus"].isna().any() or uc_storage_units["bus"].isna().any():
    raise ValueError("电源或储能包含不在最大连通图中的并网节点。")
if not np.allclose(
    uc_load_profiles.sum(axis=1),
    station_hourly_load.reindex(_snapshots).sum(axis=1) * 1000,
):
    raise ValueError("UC负荷表没有保持站点逐时负荷总量。")

## 5. 基础UC数学模型与当前约束

参考Chen等IEEE Transactions on Power Systems综述的三二进制UC结构，但不含
安全约束。目标函数包括发电边际成本、启动成本、停机成本、储能放电成本和高惩罚
弃负荷成本；`stand_by_cost=0`。

### 发电设备约束

| 设备 | 当前约束 |
|---|---|
| 煤电、天然气、其他油气、生物质、核电 | `technical_committable=True`。每类容量最大的若干节点组合使用在线、启动、停机二进制变量，满足状态转换、最小稳定出力、最大出力、最小开机及停机时间；其余组合采用连续出力。 |
| 风电、光伏 | 不做启停决策，满足\(0\le p_{g,t}\le \bar P_g a_{g,t}\)。\(a_{g,t}\)来自固定资源可用率或`cf_exp.ipynb`的逐时网格容量因子。允许弃风弃光。 |
| 径流式、水库型及其他水电 | 不做启停决策，作为连续可调发电并受逐时水文可用率上限约束；尚无库容、水量平衡、跨时段来水和末水位约束。 |
| 光热、地热 | 具有启停资格，每类仅前若干节点组合使用二进制状态；光热同时受固定资源上限约束，但尚未显式建立太阳场与热储能。 |
| 虚拟弃负荷 | 每个负荷站配置高成本正发电变量，仅用于保持问题可行，不受启停和爬坡约束。 |

对committable组合：
\[
u_{g,t}-u_{g,t-1}=v_{g,t}-w_{g,t},\qquad
P_g^{min}u_{g,t}\le p_{g,t}\le P_{g,t}^{max}u_{g,t}.
\]
最小开停机时间由PyPSA线性整数约束实现。爬坡字段保留并由
`uc_enforce_ramp_constraints`控制；当前聚合demo设为False，因为候选并网位置和
多机聚合不满足单机爬坡假设。

### 储能约束

- 充电功率和放电功率分别不超过额定功率；
- 荷电状态满足逐时能量平衡、充放电效率和standing loss；
- \(0\le SOC_{s,t}\le P_s^{nom}\times max\_hours_s\)；
- `cyclic_state_of_charge=True`，日末SOC等于日初SOC；
- 当前没有充放电互斥二进制变量、备用贡献及退化寿命成本。

### 系统与网络约束

- 每个bus、每个小时满足节点有功功率平衡；
- AC branches使用线性DC潮流、Kirchhoff电压定律和
  \(-S_l^{nom}\le f_{l,t}\le S_l^{nom}\)；
- DC branches建模为容量受限、效率为1的双向可控Link；
- 负荷是固定的站点逐时输入；
- 当前不含无功、母线电压幅值、网损、备用、惯量、排放上限、燃料供应、
  N-1、N-k及任何事故场景约束。


In [ ]:
uc_network = pypsa.Network()
uc_network.set_snapshots(_snapshots)

_carriers = sorted({
    "AC", "DC", *uc_generators["generation_type"],
    *uc_storage_units["storage_type"],
})
uc_network.add("Carrier", _carriers)
uc_network.add(
    "Bus",
    uc_buses.index,
    v_nom=uc_buses["v_nom_kv"],
    x=uc_buses["node_longitude"],
    y=uc_buses["node_latitude"],
    carrier="AC",
)
uc_network.buses["osm_node_uid"] = uc_buses["node_uid"].astype(str)
uc_network.buses["node_type"] = uc_buses["node_type"]
uc_network.buses["province_name"] = uc_buses["node_province_name"]

uc_network.add(
    "Line",
    uc_lines.index,
    bus0=uc_lines["bus0"],
    bus1=uc_lines["bus1"],
    r=uc_lines["r_ohm_equivalent"],
    x=uc_lines["x_ohm_equivalent"],
    s_nom=uc_lines["s_nom_mva"],
    length=uc_lines["length_km"],
    carrier="AC",
)
uc_network.lines["osm_branch_uid"] = uc_lines["branch_uid"].astype(str)
uc_network.lines["voltage_kv"] = uc_lines["voltage_kv"]
uc_network.lines["circuits_assumed"] = uc_lines["circuits_assumed"]

if not uc_links.empty:
    uc_network.add(
        "Link",
        uc_links.index,
        bus0=uc_links["bus0"],
        bus1=uc_links["bus1"],
        p_nom=uc_links["s_nom_mva"],
        p_min_pu=-1.0,
        p_max_pu=1.0,
        efficiency=1.0,
        carrier="DC",
    )
    uc_network.links["osm_branch_uid"] = uc_links[
        "branch_uid"
    ].astype(str)
    uc_network.links["voltage_kv"] = uc_links["voltage_kv"]

uc_network.add(
    "Generator",
    uc_generators.index,
    bus=uc_generators["bus"],
    carrier=uc_generators["generation_type"],
    p_nom=uc_generators["p_nom_mw"],
    p_min_pu=uc_generators["p_min_pu"],
    p_max_pu=uc_generator_availability,
    marginal_cost=uc_generators["marginal_cost_eur_per_mwh"],
    committable=uc_generators["committable"],
    start_up_cost=uc_generators["startup_cost_eur"],
    shut_down_cost=uc_generators["shutdown_cost_eur"],
    stand_by_cost=0.0,
    min_up_time=uc_generators["minimum_up_time_h"].fillna(0).astype(int),
    min_down_time=uc_generators[
        "minimum_down_time_h"
    ].fillna(0).astype(int),
    up_time_before=0,
    down_time_before=uc_generators[
        "minimum_down_time_h"
    ].fillna(0).astype(int),
    ramp_limit_up=(
        uc_generators["ramp_limit_up_pu"]
        if uc_enforce_ramp_constraints else np.nan
    ),
    ramp_limit_down=(
        uc_generators["ramp_limit_down_pu"]
        if uc_enforce_ramp_constraints else np.nan
    ),
    ramp_limit_start_up=1.0,
    ramp_limit_shut_down=1.0,
)
uc_network.generators["source_units"] = uc_generators["source_units"]
uc_network.generators["co2_t_per_mwh_th"] = uc_generators[
    "co2_t_per_mwh_th"
]

uc_network.add(
    "Load",
    uc_loads.index,
    bus=uc_loads["bus"],
    p_set=uc_load_profiles,
)
if not uc_storage_units.empty:
    uc_network.add(
        "StorageUnit",
        uc_storage_units.index,
        bus=uc_storage_units["bus"],
        carrier=uc_storage_units["storage_type"],
        p_nom=uc_storage_units["p_nom_mw"],
        max_hours=uc_storage_units["max_hours"],
        efficiency_store=uc_storage_units["efficiency_store"],
        efficiency_dispatch=uc_storage_units["efficiency_dispatch"],
        standing_loss=uc_storage_units["standing_loss"],
        cyclic_state_of_charge=True,
        marginal_cost=1.0,
    )

uc_network.consistency_check()
uc_model_summary = pd.Series({
    "buses": len(uc_network.buses),
    "ac_lines": len(uc_network.lines),
    "dc_links": len(uc_network.links),
    "loads": len(uc_network.loads),
    "generators": len(uc_network.generators),
    "committable_generator_clusters": (
        uc_network.generators["committable"].sum()
    ),
    "storage_units": len(uc_network.storage_units),
    "daily_load_gwh": uc_load_profiles.to_numpy().sum() / 1000,
}, name="value")
display(uc_model_summary)

## 6. 求解2024年夏季典型日

HiGHS求解24小时MILP。失负荷机组只用于保证数据不完备时模型仍有可行解；其
发电量应作为输入/网络假设不足的诊断量，而不是正常电源。

In [ ]:
uc_solver_status = uc_network.optimize(
    include_objective_constant=False,
    solver_name=uc_solver,
    solver_options={
        "time_limit": uc_solver_time_limit_s,
        "mip_rel_gap": uc_solver_mip_gap,
        "log_to_console": False,
    },
)
display(pd.Series({
    "status": uc_solver_status[0],
    "termination_condition": uc_solver_status[1],
    "objective_eur": uc_network.objective,
}, name="value"))
if uc_network.objective is None or not np.isfinite(
    uc_network.objective
):
    raise RuntimeError(
        "UC求解没有产生有限目标值，不能把当前结果解释为有效调度。"
    )


## 7. 结果汇总、绘图与PyPSA文件

彩色面积为按类型聚合的发电，黑线为系统负荷。红色`load_shedding`越大，说明
线路容量、并网映射、机组状态或容量因子假设越需要完善。NetCDF保存完整PyPSA
网络、24小时输入和优化结果，可用`pypsa.Network(path)`重新读取。

In [ ]:
uc_dispatch_by_type = (
    uc_network.generators_t.p.T
    .groupby(uc_network.generators["carrier"])
    .sum()
    .T
)
uc_total_load_mw = uc_network.loads_t.p_set.sum(axis=1)
uc_storage_discharge_mw = (
    uc_network.storage_units_t.p.clip(lower=0).sum(axis=1)
)
uc_storage_charging_mw = (
    -uc_network.storage_units_t.p.clip(upper=0).sum(axis=1)
)
uc_dispatch_by_type["storage_discharge"] = uc_storage_discharge_mw
uc_load_shedding_mwh = uc_dispatch_by_type.get(
    "load_shedding", pd.Series(0.0, index=_snapshots)
).sum()
uc_result_summary = pd.Series({
    "solver_status": uc_solver_status[0],
    "termination_condition": uc_solver_status[1],
    "objective_eur": uc_network.objective,
    "daily_load_gwh": uc_total_load_mw.sum() / 1000,
    "load_shedding_gwh": uc_load_shedding_mwh / 1000,
    "storage_discharge_gwh": uc_storage_discharge_mw.sum() / 1000,
    "storage_charging_gwh": uc_storage_charging_mw.sum() / 1000,
    "load_shedding_share_percent": (
        100 * uc_load_shedding_mwh / uc_total_load_mw.sum()
    ),
    "committed_cluster_hours": (
        uc_network.generators_t.status.fillna(0).sum().sum()
    ),
}, name="value")
display(uc_result_summary)

_dispatch_order = [
    generation_type for generation_type in [
        "nuclear", "coal", "natural_gas", "other_oil_gas",
        "bioenergy", "geothermal", "reservoir_hydropower",
        "run_of_river_hydropower", "other_hydropower",
        "onshore_wind", "offshore_wind", "utility_scale_solar",
        "solar_thermal", "storage_discharge", "load_shedding",
    ]
    if generation_type in uc_dispatch_by_type
]
_colors = {
    "nuclear": "#7B3294", "coal": "#4B5563",
    "natural_gas": "#E67E22", "other_oil_gas": "#8C564B",
    "bioenergy": "#4D9221", "geothermal": "#B8A600",
    "reservoir_hydropower": "#2166AC",
    "run_of_river_hydropower": "#4393C3",
    "other_hydropower": "#92C5DE",
    "onshore_wind": "#008B8B", "offshore_wind": "#56B4E9",
    "utility_scale_solar": "#F2C94C", "solar_thermal": "#E69F00",
    "storage_discharge": "#6A51A3",
    "load_shedding": "#D62728",
}
_figure, _axis = plt.subplots(figsize=uc_figure_size)
_ = _axis.stackplot(
    _snapshots,
    uc_dispatch_by_type[_dispatch_order].clip(lower=0).T,
    colors=[_colors[generation_type] for generation_type in _dispatch_order],
    alpha=0.88,
)
_ = _axis.plot(
    _snapshots,
    uc_total_load_mw,
    color="black",
    linewidth=1.4,
    label="Load",
    zorder=4,
)
_ = _axis.plot(
    _snapshots,
    uc_total_load_mw + uc_storage_charging_mw,
    color="#6B7280",
    linestyle="--",
    linewidth=1.0,
    label="Load + storage charging",
    zorder=4,
)
_ = _axis.set(
    title=f"China basic unit commitment demo: {uc_day.date()}",
    ylabel="Power (MW)",
    xlabel="Hour",
)
_ = _axis.grid(axis="y", color="#D1D5DB", linewidth=0.4, alpha=0.7)
_ = _axis.legend(
    handles=[
        *[
            Patch(
                facecolor=_colors[generation_type],
                label=generation_type.replace("_", " "),
            )
            for generation_type in _dispatch_order
        ],
        plt.Line2D([], [], color="black", label="load"),
        plt.Line2D(
            [], [], color="#6B7280", linestyle="--",
            label="load + storage charging",
        ),
    ],
    loc="upper left",
    bbox_to_anchor=(1.01, 1),
    frameon=False,
)
_figure.tight_layout()
_figure.savefig(
    uc_output_dir / f"uc_dispatch_{uc_day.date()}.png",
    dpi=uc_figure_dpi,
    bbox_inches="tight",
    facecolor="white",
)
plt.show()

uc_dispatch_by_type.to_csv(
    uc_output_dir / f"uc_dispatch_by_type_{uc_day.date()}.csv"
)
uc_network.generators_t.status.to_csv(
    uc_output_dir / f"uc_commitment_status_{uc_day.date()}.csv"
)
uc_network.export_to_netcdf(
    uc_output_dir / f"uc_network_{uc_day.date()}.nc"
)